# Data Collection, Understanding and Preparation

**Nguồn dữ liệu:**

| Nguồn | Vai trò | Nơi lấy |
|---|---|---|
| 311 Cases (loại `Tree Maintenance`) | Biến mục tiêu: số yêu cầu bảo trì cây | DataSF |
| Street Tree List | Đặc điểm cây theo khu phố (tính tại mốc 03/01/2022) | DataSF |
| Street-Use Permits | Nguồn phụ chính: hoạt động thi công | DataSF |
| NOAA GHCN-Daily (trạm sân bay SFO) | Nguồn mở rộng: thời tiết | NOAA |

**Phạm vi thời gian:** từ 05/01/2015 đến 11/10/2026 (614 tuần), giai đoạn mà mọi nguồn đều có số liệu thật.

**Đầu ra:**
- `SF_311_Tree_Cases_Cleaned.csv`: từng yêu cầu đã làm sạch, kèm loại yêu cầu đã chuẩn hoá và cờ đánh dấu yêu cầu trùng (dùng cho Notebook 3 và 6).
- `SF_Tree_Weekly_Data_Cleaned.csv`: bảng chính, mỗi dòng là **một khu phố trong một tuần** (dùng cho Notebook 3, 4 và 6; Notebook 3 tạo tiếp bảng đặc trưng cho Notebook 5, 7 và 8). Bảng có ba biến đếm:
  - `n`: số yêu cầu đã loại yêu cầu trùng (biến mục tiêu chính).
  - `n_all`: số yêu cầu giữ cả yêu cầu trùng.
  - `n_damaged`: số yêu cầu loại "cây hư hại" (`damaged_tree`), đã loại yêu cầu trùng.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Cấu hình chung
# TAI_LAI_DU_LIEU = False: đọc bản dữ liệu gốc đã lưu trong Data/raw/ (tải ngày 16/10/2026) để kết quả không đổi.
# TAI_LAI_DU_LIEU = True : tải lại toàn bộ từ API (số liệu có thể hơi khác do thành phố cập nhật liên tục).
TAI_LAI_DU_LIEU = False
RAW = "../Data/raw/"

MOC_KIEM_TRA_DAU = pd.Timestamp("2022-01-03")    # mốc kiểm tra đầu tiên ở Notebook 7
NGAY_DOI_HE_THONG = pd.Timestamp("2024-06-12")   # bộ phân loại 311 đổi giữa ngày 11 và 13/6/2024

## 1. Dữ liệu 311 Cases

In [ ]:
# Đọc dữ liệu 311 Cases, chỉ lấy loại yêu cầu bảo trì cây (Tree Maintenance), lấy tất cả các năm
if TAI_LAI_DU_LIEU:
    url_cases = "https://data.sfgov.org/resource/vw6y-z8j6.csv?$where=service_name=%27Tree%20Maintenance%27&$limit=1000000"
    df_cases = pd.read_csv(url_cases)
    df_cases.to_csv(RAW + "raw_311_tree_cases.csv", index=False)   # lưu bản gốc
else:
    df_cases = pd.read_csv(RAW + "raw_311_tree_cases.csv")

print(f"Number of rows    : {df_cases.shape[0]:,}")
print(f"Number of columns : {df_cases.shape[1]}")

Number of rows    : 176,606
Number of columns : 25


In [ ]:
# kiểm tra thông tin dữ liệu
print(df_cases.info())  # Kiểm tra kiểu dữ liệu
print(df_cases.head())  # Xem trước 5 dòng đầu

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 176606 entries, 0 to 176605
Data columns (total 25 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   service_request_id               176606 non-null  int64  
 1   requested_datetime               176606 non-null  object 
 2   closed_date                      148537 non-null  object 
 3   updated_datetime                 176606 non-null  object 
 4   status_description               176606 non-null  object 
 5   status_notes                     174334 non-null  object 
 6   agency_responsible               176606 non-null  object 
 7   service_name                     176606 non-null  object 
 8   service_subtype                  176606 non-null  object 
 9   service_details                  176364 non-null  object 
 10  address                          176605 non-null  object 
 11  street                           175235 non-null  object 
 12  su

In [ ]:
# Đổi tên cột khu phố và chuyển cột thời gian yêu cầu về dạng datetime
df_cases = df_cases.rename(columns={"analysis_neighborhood": "nbhd"})
df_cases["requested_datetime"] = pd.to_datetime(df_cases["requested_datetime"], errors="coerce")
df_cases[["nbhd", "requested_datetime", "service_name"]].head()

,nbhd,requested_datetime,service_name
0,Inner Richmond,2021-01-19 06:52:00,Tree Maintenance
1,Lone Mountain/USF,2021-02-04 10:36:00,Tree Maintenance
2,North Beach,2021-01-22 18:00:00,Tree Maintenance
3,Hayes Valley,2021-01-19 10:44:00,Tree Maintenance
4,Mission,2021-01-23 16:08:00,Tree Maintenance


In [ ]:
# kiểm tra dữ liệu thiếu
missing_values = df_cases.isnull().sum()
missing_percentage = (df_cases.isnull().mean() * 100)
missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage (%)': missing_percentage
})

missing_df = missing_df[missing_df['Missing Values'] > 0].sort_values(by='Percentage (%)', ascending=False)
print(missing_df)

                                 Missing Values  Percentage (%)
media_url                                118996       67.379364
bos_2012                                  45072       25.521217
closed_date                               28069       15.893571
nbhd                                      11769        6.663987
status_notes                               2272        1.286480
police_district                            1421        0.804616
street                                     1371        0.776304
neighborhoods_sffind_boundaries            1346        0.762149
supervisor_district                        1262        0.714585
source                                      397        0.224794
service_details                             242        0.137028
point                                         6        0.003397
lat                                           6        0.003397
long                                          6        0.003397
point_geom                              

In [ ]:
# Bỏ các dòng không xác định được khu phố hoặc thời gian yêu cầu (không thể xếp vào bảng khu phố - tuần)
n0 = len(df_cases)
df_cases = df_cases.dropna(subset=["nbhd", "requested_datetime"])
print(f"Đã bỏ {n0 - len(df_cases)} dòng thiếu khu phố hoặc thời gian yêu cầu")

Đã bỏ 11769 dòng thiếu khu phố hoặc thời gian yêu cầu


In [ ]:
# Kiểm tra trùng lặp theo mã yêu cầu (một yêu cầu bị ghi 2 lần)
duplicates = df_cases.duplicated(subset=["service_request_id"]).sum()
print("Duplicate rows:", duplicates)
df_cases = df_cases.drop_duplicates(subset=["service_request_id"])

# Đánh dấu các yêu cầu bị chính thành phố đánh dấu là trùng ("Case is a Duplicate"):
# thường là nhiều người cùng báo một cây, giữ lại sẽ làm số nhu cầu bị phóng đại.
# Nhãn này chỉ có sau khi yêu cầu được xử lý, nên không bỏ hẳn mà giữ lại cờ la_trung
# để tạo hai phiên bản biến mục tiêu: n (đã loại trùng) và n_all (giữ cả trùng).
df_cases["la_trung"] = df_cases["status_notes"].str.contains("duplicate", case=False, na=False)
print(f"Số yêu cầu bị đánh dấu trùng: {df_cases['la_trung'].sum()} ({df_cases['la_trung'].mean() * 100:.2f}%)")
print(df_cases.loc[df_cases["la_trung"], "status_notes"].str[:80].value_counts().head(10))

Duplicate rows: 0
Số yêu cầu bị đánh dấu trùng: 16533 (10.03%)
status_notes
Case is a Duplicate - Due to the implementation of Prop-E, resources are being f    4812
Case is a Duplicate                                                                 4132
Case is a Duplicate - Duplicate                                                     1207
Case is a Duplicate - Request is a duplicate and has been previously reported as    1204
Case Resolved - duplicate                                                            867
Case is a Duplicate -  Due to the implementation of Prop-E, resources are being      454
Case is a Duplicate - Action Reason: Duplicate                                       342
case_is_a_duplicate - Duplicate                                                      287
case_is_a_duplicate - Action Reason: Duplicate                                       249
Case is a Duplicate -                                                                 90
Name: count, dtype: int64


In [ ]:
# Tạo các biến thời gian
df_cases["date"] = df_cases["requested_datetime"].dt.floor("D")
df_cases["week_start"] = df_cases["date"] - pd.to_timedelta(df_cases["date"].dt.dayofweek, unit="D")  # Thứ Hai đầu tuần
df_cases["year"] = df_cases["date"].dt.year
df_cases["month"] = df_cases["date"].dt.month
df_cases["dow"] = df_cases["date"].dt.dayofweek   # 0 = Thứ Hai, ..., 6 = Chủ Nhật

print(f"Khoảng thời gian : {df_cases['date'].min().date()} đến {df_cases['date'].max().date()}")
print(f"Số khu phố       : {df_cases['nbhd'].nunique()}")

Khoảng thời gian : 2008-07-01 đến 2026-10-14
Số khu phố       : 41


In [ ]:
print(f"Số yêu cầu còn lại (giữ cả trùng): {len(df_cases):,}")
print(f"Số yêu cầu còn lại (đã loại trùng): {(~df_cases['la_trung']).sum():,}")

Số yêu cầu còn lại (giữ cả trùng): 164,837
Số yêu cầu còn lại (đã loại trùng): 148,304


### Chuẩn hoá loại yêu cầu

Từ giữa tháng 6/2024, hệ thống 311 đổi cách ghi loại yêu cầu (`service_subtype`): bộ phân loại cũ có dạng `Trees - Damaged_Tree`, bộ phân loại mới có dạng `damaged_tree`. Để so sánh được qua hai giai đoạn, các tên được đưa về cùng một bộ năm loại: `damaged_tree`, `overgrown_tree`, `damaging_property`, `landscaping`, `other`.

In [ ]:
# Đưa tên loại yêu cầu về chữ thường, bỏ tiền tố "Trees - " của bộ phân loại cũ
loai = df_cases["service_subtype"].str.lower().str.replace("trees - ", "", regex=False).str.strip()

# Gộp các loại nhỏ hoặc không rõ ràng vào nhóm gần nhất
loai = loai.replace({
    "damaging_property_roots": "damaging_property",
    "property_damage": "damaging_property",
    "tree_other": "other",
    "other_enter_additional_details_below": "other",
    "city_attorney": "other",
    "public_works": "other",
    "na": "other",
})
df_cases["loai_yeu_cau"] = loai

# Cờ cho biết yêu cầu được ghi theo bộ phân loại mới (từ ngày đổi hệ thống trở đi)
df_cases["he_thong_moi"] = df_cases["date"] >= NGAY_DOI_HE_THONG

# Kiểm tra: ngày cuối cùng còn tên kiểu cũ và ngày đầu tiên có tên kiểu mới
kieu_moi = df_cases["service_subtype"].str.match(r"^[a-z]", na=False)
print("Ngày cuối có tên kiểu cũ :", df_cases.loc[~kieu_moi, "date"].max().date())
print("Ngày đầu có tên kiểu mới :", df_cases.loc[kieu_moi, "date"].min().date())
print()
print(pd.crosstab(df_cases["loai_yeu_cau"], df_cases["he_thong_moi"].map({False: "Trước 6/2024", True: "Từ 6/2024"})))

Ngày cuối có tên kiểu cũ : 2024-06-11
Ngày đầu có tên kiểu mới : 2024-06-13

he_thong_moi       Trước 6/2024  Từ 6/2024
loai_yeu_cau                              
damaged_tree              48829      12309
damaging_property         20256       4782
landscaping               16579       7059
other                     12247        383
overgrown_tree            30882      11511


## 2. Dữ liệu Street-Use Permits

In [ ]:
# Đọc dữ liệu SF Street-Use Permits: lấy toàn bộ giấy phép (mọi năm), chỉ chọn 4 cột cần dùng vì dữ liệu gốc quá lớn (31 cột)
if TAI_LAI_DU_LIEU:
    url_permits = ("https://data.sfgov.org/resource/b6tj-gt35.csv"
                   "?$select=permit_number,permit_start_date,permit_end_date,analysis_neighborhood"
                   "&$limit=2000000")
    df_permits = pd.read_csv(url_permits)
    df_permits.to_csv(RAW + "raw_street_use_permits.csv", index=False)   # lưu bản gốc
else:
    df_permits = pd.read_csv(RAW + "raw_street_use_permits.csv")

df_permits = df_permits.rename(columns={"analysis_neighborhood": "nbhd"})
print(f"Number of rows    : {df_permits.shape[0]:,}")
print(f"Number of columns : {df_permits.shape[1]}")

Number of rows    : 1,233,168
Number of columns : 4


In [ ]:
# Chuyển đổi ngày bắt đầu / kết thúc giấy phép
df_permits["permit_start_date"] = pd.to_datetime(df_permits["permit_start_date"], errors="coerce")
df_permits["permit_end_date"] = pd.to_datetime(df_permits["permit_end_date"], errors="coerce")

# Kiểm tra dữ liệu thiếu ở từng cột
print(df_permits.isnull().sum())

# Bỏ giấy phép thiếu khu phố, ngày bắt đầu hoặc ngày kết thúc
# (giấy phép thiếu ngày kết thúc sẽ bị tính là "đang hoạt động mãi mãi", làm sai số liệu)
n0 = len(df_permits)
df_permits = df_permits.dropna(subset=["nbhd", "permit_start_date", "permit_end_date"])
print(f"Đã bỏ {n0 - len(df_permits)} dòng thiếu khu phố hoặc ngày")

# Bỏ giấy phép có ngày kết thúc trước ngày bắt đầu (lỗi nhập liệu)
n0 = len(df_permits)
df_permits = df_permits[df_permits["permit_end_date"] >= df_permits["permit_start_date"]]
print(f"Đã bỏ {n0 - len(df_permits)} dòng có ngày kết thúc trước ngày bắt đầu")

# Bỏ giấy phép trùng: một giấy phép trải nhiều đoạn phố được ghi thành nhiều dòng,
# chỉ giữ 1 dòng cho mỗi giấy phép trong mỗi khu phố
n0 = len(df_permits)
df_permits = df_permits.drop_duplicates(subset=["permit_number", "nbhd"])
print(f"Đã bỏ {n0 - len(df_permits)} dòng trùng")
print(f"Số giấy phép còn lại: {len(df_permits):,}")

permit_number             0
permit_start_date    502768
permit_end_date      509569
nbhd                    502
dtype: int64
Đã bỏ 517549 dòng thiếu khu phố hoặc ngày
Đã bỏ 3916 dòng có ngày kết thúc trước ngày bắt đầu
Đã bỏ 352813 dòng trùng
Số giấy phép còn lại: 358,890


Nhận xét: Có khoảng 42% giấy phép bị bỏ do dữ liệu gốc không ghi ngày bắt đầu hoặc ngày kết thúc (đã kiểm tra lại trên file gốc, không phải do lỗi đọc ngày). Không có ngày thì không xác định được giấy phép hoạt động trong khoảng thời gian nào, nên không thể dùng. Khoảng 50% số dòng còn lại bị trùng vì một giấy phép trải trên nhiều đoạn phố được ghi thành nhiều dòng; mỗi giấy phép chỉ giữ một dòng cho mỗi khu phố. Kết quả còn 358,890 giấy phép.

## 3. Dữ liệu thời tiết NOAA

In [ ]:
# Đọc dữ liệu thời tiết NOAA GHCN-Daily, trạm sân bay San Francisco (nguồn mở rộng), lấy từ 2008
if TAI_LAI_DU_LIEU:
    url_weather = ("https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries"
                   "&dataTypes=TMAX,TMIN,PRCP,AWND&stations=USW00023234"
                   "&startDate=2008-01-01&endDate=2026-10-16&format=csv&units=metric")
    df_weather = pd.read_csv(url_weather)
    df_weather.to_csv(RAW + "raw_noaa_weather.csv", index=False)   # lưu bản gốc
else:
    df_weather = pd.read_csv(RAW + "raw_noaa_weather.csv")

df_weather["DATE"] = pd.to_datetime(df_weather["DATE"], errors="coerce")
df_weather = df_weather.rename(columns={"DATE": "date"})
print(f"Number of rows    : {df_weather.shape[0]:,}")
print(f"Khoảng thời gian  : {df_weather['date'].min().date()} đến {df_weather['date'].max().date()}")
df_weather[["date", "PRCP", "AWND", "TMAX"]].describe()

Number of rows    : 6,859
Khoảng thời gian  : 2008-01-01 đến 2026-10-11


,date,PRCP,AWND,TMAX
count,6859,6859.000000,6857.000000,6839.000000
mean,2017-05-22 00:00:00,1.219711,4.586204,19.294853
min,2008-01-01 00:00:00,0.000000,0.200000,7.200000
25%,2012-09-10 12:00:00,0.000000,3.100000,16.100000
50%,2017-05-22 00:00:00,0.000000,4.600000,18.900000
75%,2022-01-30 12:00:00,0.000000,5.900000,22.200000
max,2026-10-11 00:00:00,102.100000,13.500000,40.000000
std,NaN,5.193990,2.036898,4.569724


## 4. Dữ liệu Street Tree List

In [ ]:
# Đọc dữ liệu Street Tree List (danh mục cây đường phố hiện tại), lấy tất cả các cây
if TAI_LAI_DU_LIEU:
    url_trees = "https://data.sfgov.org/resource/tkzw-k3nq.csv?$limit=1000000"
    df_trees = pd.read_csv(url_trees)
    df_trees.to_csv(RAW + "raw_street_tree_list.csv", index=False)   # lưu bản gốc
else:
    df_trees = pd.read_csv(RAW + "raw_street_tree_list.csv")

df_trees = df_trees.rename(columns={"analysis_neighborhood": "nbhd"})
print(f"Number of rows    : {df_trees.shape[0]:,}")
print(f"Number of columns : {df_trees.shape[1]}")

Number of rows    : 144,491
Number of columns : 27


In [ ]:
# Chuyển kiểu dữ liệu và bỏ cây không có khu phố
df_trees["planteddate"] = pd.to_datetime(df_trees["planteddate"], errors="coerce")
df_trees["mapdbh"] = pd.to_numeric(df_trees["mapdbh"], errors="coerce")

n0 = len(df_trees)
df_trees = df_trees.dropna(subset=["nbhd"])
print(f"Đã bỏ {n0 - len(df_trees)} cây không có khu phố")

# Đường kính thân (dbh, inch) bằng 0 hoặc âm là không hợp lệ -> coi là thiếu
df_trees.loc[df_trees["mapdbh"] <= 0, "mapdbh"] = np.nan

# Ngày trồng sau ngày tải dữ liệu là lỗi -> coi là thiếu
df_trees.loc[df_trees["planteddate"] > pd.Timestamp("2026-10-16"), "planteddate"] = pd.NaT

co_ngay = df_trees["planteddate"].notna()
print(f"Số cây có ghi ngày trồng : {co_ngay.sum():,} ({co_ngay.mean() * 100:.1f}%)")
print(f"Trong đó trồng từ mốc kiểm tra đầu tiên ({MOC_KIEM_TRA_DAU.date()}) trở đi: "
      f"{(df_trees['planteddate'] >= MOC_KIEM_TRA_DAU).sum():,}")

df_trees[["mapdbh"]].describe()

Đã bỏ 5756 cây không có khu phố
Số cây có ghi ngày trồng : 33,735 (24.3%)
Trong đó trồng từ mốc kiểm tra đầu tiên (2022-01-03) trở đi: 5,965


,mapdbh
count,130387.000000
mean,9.391067
std,9.136145
min,3.000000
25%,3.000000
50%,7.000000
75%,12.000000
max,130.000000


In [ ]:
# Tổng hợp đặc điểm cây theo khu phố (dùng trung vị để không bị ảnh hưởng bởi vài giá trị quá lớn)
# Street Tree List là danh mục hiện tại, không có lịch sử. Để không dùng thông tin của giai đoạn kiểm tra,
# đặc điểm cây được tính tại mốc kiểm tra đầu tiên (03/01/2022):
#  - chỉ giữ cây không ghi ngày trồng (coi là đã có từ trước) hoặc trồng trước mốc này
#  - tuổi cây tính từ ngày trồng đến mốc này (chỉ tính được cho cây có ghi ngày trồng)
# Mỗi khu phố một giá trị, dùng chung cho mọi tuần.
trees_moc = df_trees[df_trees["planteddate"].isna() | (df_trees["planteddate"] < MOC_KIEM_TRA_DAU)].copy()
trees_moc["tuoi_cay"] = (MOC_KIEM_TRA_DAU - trees_moc["planteddate"]).dt.days / 365.25

tree_features = trees_moc.groupby("nbhd").agg(
    so_cay=("treeid", "count"),
    dbh_tb=("mapdbh", "median"),
    tuoi_cay_tb=("tuoi_cay", "median"),
).reset_index()

print(f"Số cây dùng để tính đặc điểm: {len(trees_moc):,} / {len(df_trees):,}")
print("Số khu phố có dữ liệu cây:", tree_features["nbhd"].nunique())
tree_features.sort_values("so_cay", ascending=False).head()

Số cây dùng để tính đặc điểm: 132,770 / 138,735
Số khu phố có dữ liệu cây: 41


,nbhd,so_cay,dbh_tb,tuoi_cay_tb
39,West of Twin Peaks,10548,7.0,15.764545
34,Sunset/Parkside,10421,4.0,15.060917
18,Mission,9115,7.0,12.372348
0,Bayview Hunters Point,9030,6.0,15.816564
25,Outer Richmond,5717,5.0,13.850787


Nhận xét: Chỉ khoảng 24% số cây có ghi ngày trồng, nên không thể dựng lại số cây của từng tuần một cách đáng tin cậy. Vì vậy đặc điểm cây được tính một lần tại mốc kiểm tra đầu tiên (03/01/2022) và dùng chung cho mọi tuần: các cây trồng từ mốc này trở đi bị loại ra, để mô hình không biết trước số cây của giai đoạn kiểm tra. Tuổi cây trung vị chỉ tính trên phần nhỏ số cây có ngày trồng và có thể bị lệch (cây mới trồng thường có hồ sơ đầy đủ hơn); đường kính thân (`dbh`) là số đo hiện tại. Nhóm đặc trưng này cần được diễn giải thận trọng, và Notebook 7 có thử nghiệm bỏ hẳn nhóm này.

## 5. Lọc về cùng một khoảng thời gian

Mọi nguồn chỉ dùng dữ liệu **từ năm 2015**. Mỗi tuần tính từ Thứ Hai đến Chủ Nhật.
- Tuần đầu tiên: Thứ Hai đầu tiên của năm 2015.
- Tuần cuối cùng: tuần đầy đủ cuối cùng mà **cả 311 và thời tiết** đều có số liệu (thời tiết NOAA thường cập nhật chậm hơn vài ngày).

In [ ]:
first_week = pd.Timestamp("2015-01-05")   # Thứ Hai đầu tiên của năm 2015

end_data = min(df_cases["date"].max(), df_weather["date"].max())
last_week = end_data - pd.Timedelta(days=6)
last_week = last_week - pd.Timedelta(days=last_week.dayofweek)   # lùi về Thứ Hai
last_day = last_week + pd.Timedelta(days=6)                      # Chủ Nhật cuối cùng

weeks = pd.date_range(first_week, last_week, freq="W-MON")
print(f"Từ tuần {first_week.date()} đến hết ngày {last_day.date()} -> {len(weeks)} tuần")

Từ tuần 2015-01-05 đến hết ngày 2026-10-11 -> 614 tuần


In [ ]:
# Lọc 311
n0 = len(df_cases)
df_cases = df_cases[(df_cases["date"] >= first_week) & (df_cases["date"] <= last_day)]
print(f"311: đã bỏ {n0 - len(df_cases):,} yêu cầu ngoài khoảng thời gian, còn {len(df_cases):,}")

# Lọc thời tiết
df_weather = df_weather[(df_weather["date"] >= first_week) & (df_weather["date"] <= last_day)]
print(f"Thời tiết: còn {len(df_weather):,} ngày")

# Lọc giấy phép: bỏ giấy phép không hoạt động ngày nào trong giai đoạn nghiên cứu.
# Giấy phép bắt đầu trước 2015 nhưng còn hiệu lực sang 2015 thì vẫn giữ, coi như bắt đầu từ tuần đầu tiên.
n0 = len(df_permits)
df_permits = df_permits[(df_permits["permit_end_date"] >= first_week) & (df_permits["permit_start_date"] <= last_day)].copy()
df_permits["permit_start_date"] = df_permits["permit_start_date"].clip(lower=first_week)
print(f"Giấy phép: đã bỏ {n0 - len(df_permits):,} giấy phép ngoài khoảng thời gian, còn {len(df_permits):,}")

311: đã bỏ 32,641 yêu cầu ngoài khoảng thời gian, còn 132,196
Thời tiết: còn 4,298 ngày
Giấy phép: đã bỏ 151,433 giấy phép ngoài khoảng thời gian, còn 207,457


## 6. Gom dữ liệu theo khu phố và tuần

In [ ]:
# Số yêu cầu bảo trì cây của mỗi khu phố trong mỗi tuần
#  n         : đã loại yêu cầu trùng (biến mục tiêu chính)
#  n_all     : giữ cả yêu cầu trùng
#  n_damaged : chỉ loại "cây hư hại" (damaged_tree), đã loại yêu cầu trùng
nbhds = sorted(df_cases["nbhd"].unique())
full_idx = pd.MultiIndex.from_product([nbhds, weeks], names=["nbhd", "week_start"])

def dem_theo_tuan(d, ten):
    # Lấp đầy lưới: mỗi khu phố phải có đủ mọi tuần (tuần không có yêu cầu thì = 0)
    return d.groupby(["nbhd", "week_start"]).size().rename(ten).reindex(full_idx, fill_value=0)

khong_trung = df_cases[~df_cases["la_trung"]]
weekly = pd.concat([
    dem_theo_tuan(khong_trung, "n"),
    dem_theo_tuan(df_cases, "n_all"),
    dem_theo_tuan(khong_trung[khong_trung["loai_yeu_cau"] == "damaged_tree"], "n_damaged"),
], axis=1).reset_index()

print(f"Số dòng: {len(weekly):,} ({len(nbhds)} khu phố x {len(weeks)} tuần)")
for col in ["n", "n_all", "n_damaged"]:
    print(f"{col:10s}: tổng {weekly[col].sum():,} yêu cầu, tỷ lệ tuần bằng 0 = {(weekly[col] == 0).mean() * 100:.1f}%")

Số dòng: 25,174 (41 khu phố x 614 tuần)
n         : tổng 116,057 yêu cầu, tỷ lệ tuần bằng 0 = 19.4%
n_all     : tổng 132,196 yêu cầu, tỷ lệ tuần bằng 0 = 18.0%
n_damaged : tổng 44,737 yêu cầu, tỷ lệ tuần bằng 0 = 39.6%


In [ ]:
# Xây dựng biến permits_active: số giấy phép đang hoạt động tại mỗi khu phố, mỗi ngày
# Mỗi giấy phép +1 vào ngày bắt đầu, -1 vào ngày sau ngày kết thúc, rồi cộng dồn theo ngày
starts = df_permits[["nbhd", "permit_start_date"]].rename(columns={"permit_start_date": "date"})
starts["delta"] = 1

ends = df_permits[["nbhd", "permit_end_date"]].rename(columns={"permit_end_date": "date"})
ends["date"] = ends["date"] + pd.Timedelta(days=1)
ends["delta"] = -1

events = pd.concat([starts, ends])
events["date"] = events["date"].dt.floor("D")
events = events.groupby(["nbhd", "date"])["delta"].sum()

days = pd.date_range(first_week, last_day, freq="D")
full_idx2 = pd.MultiIndex.from_product([nbhds, days], names=["nbhd", "date"])
permits_daily = events.reindex(full_idx2, fill_value=0).reset_index()
permits_daily["permits_active"] = permits_daily.groupby("nbhd")["delta"].cumsum()

# Đổi sang tuần: lấy trung bình số giấy phép đang hoạt động trong 7 ngày
permits_daily["week_start"] = permits_daily["date"] - pd.to_timedelta(permits_daily["date"].dt.dayofweek, unit="D")
permits_weekly = permits_daily.groupby(["nbhd", "week_start"])["permits_active"].mean().reset_index()

print(permits_weekly["permits_active"].describe())

count    25174.000000
mean        80.890227
std         86.007861
min          0.000000
25%         18.142857
50%         59.571429
75%        115.428571
max        612.714286
Name: permits_active, dtype: float64


In [ ]:
# Thời tiết theo tuần (một trạm duy nhất nên dùng chung cho cả 41 khu phố)
# PRCP: tổng lượng mưa trong tuần | AWND: gió trung bình ngày cao nhất trong tuần | TMAX: nhiệt độ cao nhất trung bình
df_weather["week_start"] = df_weather["date"] - pd.to_timedelta(df_weather["date"].dt.dayofweek, unit="D")
weather_weekly = df_weather.groupby("week_start").agg(
    PRCP=("PRCP", "sum"),
    AWND=("AWND", "max"),
    TMAX=("TMAX", "mean"),
).reset_index()

print(f"Số tuần có dữ liệu thời tiết: {len(weather_weekly)} / {len(weeks)}")

Số tuần có dữ liệu thời tiết: 614 / 614


## 7. Ghép các nguồn thành bảng chính

In [ ]:
df = weekly.merge(permits_weekly, on=["nbhd", "week_start"], how="left")
df = df.merge(weather_weekly, on="week_start", how="left")
df = df.merge(tree_features, on="nbhd", how="left")

# Kiểm tra tên khu phố giữa 311 và Street Tree List có khớp nhau không
khong_khop = set(nbhds) - set(tree_features["nbhd"])
print("Khu phố có trong 311 nhưng không có trong Street Tree List:", khong_khop if khong_khop else "không có")

Khu phố có trong 311 nhưng không có trong Street Tree List: không có


In [ ]:
# Tạo đặc trưng thời gian và ép kiểu dữ liệu phân loại
df["year"] = df["week_start"].dt.year
df["month"] = df["week_start"].dt.month
df["week_of_year"] = df["week_start"].dt.isocalendar().week.astype(int)
df["nbhd"] = df["nbhd"].astype("category")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25174 entries, 0 to 25173
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   nbhd            25174 non-null  category      
 1   week_start      25174 non-null  datetime64[ns]
 2   n               25174 non-null  int64         
 3   n_all           25174 non-null  int64         
 4   n_damaged       25174 non-null  int64         
 5   permits_active  25174 non-null  float64       
 6   PRCP            25174 non-null  float64       
 7   AWND            25174 non-null  float64       
 8   TMAX            25092 non-null  float64       
 9   so_cay          25174 non-null  int64         
 10  dbh_tb          24560 non-null  float64       
 11  tuoi_cay_tb     25174 non-null  float64       
 12  year            25174 non-null  int32         
 13  month           25174 non-null  int32         
 14  week_of_year    25174 non-null  int64         
dtypes:

In [ ]:
# Kiểm tra missing còn lại sau khi ghép
missing = df.isnull().sum()
missing = missing[missing > 0]
print(missing.sort_values(ascending=False))

# Thời tiết thiếu (trạm không ghi số liệu): điền bằng trung vị của cùng tháng (thời tiết thay đổi theo mùa).
# Trung vị chỉ tính trên các tuần trước mốc kiểm tra đầu tiên, để không dùng thông tin của giai đoạn kiểm tra.
truoc_moc = df["week_start"] < MOC_KIEM_TRA_DAU
for col in ["PRCP", "AWND", "TMAX"]:
    trung_vi_thang = df.loc[truoc_moc].groupby("month")[col].median()
    df[col] = df[col].fillna(df["month"].map(trung_vi_thang))

# Đặc điểm cây thiếu: điền bằng trung vị của các khu phố khác (đặc điểm cây đã cố định tại mốc kiểm tra đầu tiên)
for col in ["so_cay", "dbh_tb", "tuoi_cay_tb"]:
    df[col] = df[col].fillna(df[col].median())

dbh_tb    614
TMAX       82
dtype: int64


In [ ]:
# kiểm tra dữ liệu sau khi làm sạch
print(df.info())
print(f"Khoảng thời gian : {df['week_start'].min().date()} đến {df['week_start'].max().date()}")
print(f"Số khu phố       : {df['nbhd'].nunique()}")
print(f"Số tuần          : {df['week_start'].nunique()}")
print(f"Tổng số yêu cầu n     : {df['n'].sum():,} (phải bằng số yêu cầu 311 đã loại trùng: {(~df_cases['la_trung']).sum():,})")
print(f"Tổng số yêu cầu n_all : {df['n_all'].sum():,} (phải bằng số yêu cầu 311 giữ cả trùng: {len(df_cases):,})")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25174 entries, 0 to 25173
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   nbhd            25174 non-null  category      
 1   week_start      25174 non-null  datetime64[ns]
 2   n               25174 non-null  int64         
 3   n_all           25174 non-null  int64         
 4   n_damaged       25174 non-null  int64         
 5   permits_active  25174 non-null  float64       
 6   PRCP            25174 non-null  float64       
 7   AWND            25174 non-null  float64       
 8   TMAX            25174 non-null  float64       
 9   so_cay          25174 non-null  int64         
 10  dbh_tb          25174 non-null  float64       
 11  tuoi_cay_tb     25174 non-null  float64       
 12  year            25174 non-null  int32         
 13  month           25174 non-null  int32         
 14  week_of_year    25174 non-null  int64         
dtypes:

In [ ]:
df_cases[["service_request_id", "requested_datetime", "date", "week_start", "year", "month", "dow",
          "nbhd", "service_subtype", "loai_yeu_cau", "he_thong_moi", "la_trung"]].to_csv(
    "../Data/processed/SF_311_Tree_Cases_Cleaned.csv", index=False)
df.to_csv("../Data/processed/SF_Tree_Weekly_Data_Cleaned.csv", index=False)
print("Dữ liệu đã được làm sạch và lưu thành công!")

Dữ liệu đã được làm sạch và lưu thành công!
